# Sesión 3 · Grupos sin etiqueta
**Machine Learning y Seguridad en la Nube — Especialización en Big Data (UCompensar)**

Este notebook acompaña las [diapositivas de la sesión](https://jotamao1985.github.io/Machine-Learning-cloud-computing_Ucompensar/diapositivas/sesion-3-grupos-sin-etiqueta.html).
Cada celda lleva entre corchetes el mismo identificador que el código proyectado, así que puedes seguir
la clase celda a celda. La portada del curso está en https://jotamao1985.github.io/Machine-Learning-cloud-computing_Ucompensar/
y el enunciado del proyecto final, en [su página](https://jotamao1985.github.io/Machine-Learning-cloud-computing_Ucompensar/proyecto.html).

1. **Archivo → Guardar una copia en Drive** para que tus cambios no se pierdan.
2. Ejecuta las celdas **en orden** con **Mayús + Enter**. Varias usan resultados de las anteriores.
   Si Colab avisa que el notebook no es de Google, elige **Ejecutar de todos modos**.
3. Si Colab se reinicia o una celda falla al repetirla, usa **Entorno de ejecución → Ejecutar anteriores**.
   Las celdas de ejercicio dan error mientras tengan `...` sin reemplazar (si el error menciona `Ellipsis` o
   `'ellipsis'`, o el verificador dice que quedan puntos suspensivos, te quedó un `...`). Colab se detiene en el primer error: completa los ejercicios en orden, o ejecuta
   con **Mayús + Enter** solo las celdas que necesites.
4. La primera celda **descarga** los datos del caso *Yarumo Fibra S.A.S.*, una empresa **ficticia** (sus
   datos se generaron para el curso y no corresponden a personas ni a empresas reales), y comprueba que
   el archivo sea idéntico al del curso. Si la descarga falla, la celda te dice cómo subirlo a mano.

En el código, los decimales van con punto (así los escribe Python); en el texto, con coma. Todo el
notebook tarda menos de un minuto en ejecutarse.

In [ ]:
#@title ▶ Descarga los datos del caso (ejecútala sin modificar) { display-mode: "form" }
"""Datos del caso «Yarumo Fibra S.A.S.» (empresa FICTICIA), descargados del repositorio del curso.

Deja en datos/clientes.csv los 8000 clientes del curso y comprueba su huella SHA-256: si el
archivo no es idéntico al del curso, no se usa. Solo usa la biblioteca estándar de Python.
"""

import hashlib
import os
import urllib.request
from pathlib import Path

DATOS_URL = "https://raw.githubusercontent.com/JotaMao1985/Machine-Learning-cloud-computing_Ucompensar/main/datos/clientes.csv"
DATOS_SHA256 = "2d4fad61a49efcf1d071b6c412e5224e71eca23047cbbac6b4017c8627f9c93b"
PORTADA = "https://jotamao1985.github.io/Machine-Learning-cloud-computing_Ucompensar/"


def _es_del_curso(contenido):
    return hashlib.sha256(contenido).hexdigest() == DATOS_SHA256


def _sin_datos(motivo):
    return (f"No pude dejar listos los datos: {motivo}.\n"
            f"Descarga clientes.csv desde la portada del curso ({PORTADA}), súbelo a Colab con el "
            "ícono de carpeta de la izquierda (Subir al almacenamiento de la sesión) y vuelve a "
            "ejecutar esta celda.")


def _leer_url():
    # Solo al preparar el material: la variable apunta a una copia local del mismo archivo
    local = os.environ.get("YARUMO_CSV_LOCAL")
    if local:
        return Path(local).read_bytes()
    with urllib.request.urlopen(DATOS_URL, timeout=30) as respuesta:
        return respuesta.read()


def descargar_datos(carpeta="datos"):
    """Deja <carpeta>/clientes.csv, idéntico al del curso, y devuelve su ruta."""
    ruta = Path(carpeta) / "clientes.csv"
    subido = Path("clientes.csv")  # por si se subió a mano a la carpeta de trabajo de Colab
    if ruta.exists() and _es_del_curso(ruta.read_bytes()):
        contenido, origen = ruta.read_bytes(), "ya estaban"
    elif subido.exists() and _es_del_curso(subido.read_bytes()):
        contenido, origen = subido.read_bytes(), "copiados del archivo que subiste"
    else:
        try:
            contenido = _leer_url()
        except OSError as error:
            raise RuntimeError(_sin_datos(f"no se pudo descargar el archivo ({error})")) from None
        if not _es_del_curso(contenido):
            raise RuntimeError(_sin_datos("el archivo descargado no es el del curso"))
        origen = "descargados del repositorio del curso"
    if origen != "ya estaban":
        ruta.parent.mkdir(parents=True, exist_ok=True)
        temporal = ruta.with_suffix(".tmp")  # sin archivos a medias si algo falla al escribir
        temporal.write_bytes(contenido)
        temporal.replace(ruta)
    lineas = contenido.decode("utf-8").splitlines()
    print(f"Listo: {ruta} con {len(lineas) - 1} clientes y {len(lineas[0].split(','))} columnas, {origen}")
    return ruta


"""Ayudas comunes a todos los notebooks del curso (van en la celda oculta, tras descarga_datos.py).

Las de cada sesión están en ayudas_sesionN.py.
"""


def _aviso_version():
    """Las diapositivas se ejecutaron con pandas 2.2 y scikit-learn 1.6: si Colab trae otras, se dice."""
    import pandas as pd
    import sklearn

    if pd.__version__.split(".")[:2] != ["2", "2"] or sklearn.__version__.split(".")[:2] != ["1", "6"]:
        print(f"Nota: este Colab trae pandas {pd.__version__} y scikit-learn {sklearn.__version__}; "
              "las diapositivas se hicieron con pandas 2.2 y scikit-learn 1.6. Si alguna salida "
              "se ve un poco distinta, es por eso.")


_aviso_version()


"""Sesión 3: estilo de las gráficas del curso y los verificadores de la práctica.

Van dentro de la celda oculta del notebook, junto con descarga_datos.py y ayudas.py. Cada
verificador recibe lo que el estudiante calculó, dice qué está bien y, en lo demás, da una
pista sin la respuesta. Nunca se cae: un «...» sin reemplazar o un tipo inesperado se avisa.

Los valores esperados se calculan aquí, en tiempo de ejecución, desde datos/clientes.csv, con
las mismas columnas, el mismo escalado y la misma semilla de la clase (y se guardan para no
repetir el cálculo). Los grupos se reconocen por su perfil (edad, usuarios, pymes), nunca por el
número de etiqueta, que puede cambiar con otra versión de scikit-learn.
Tolerancias: ejercicio 1, ±0,005 en cada silueta; ejercicio 2, ±1,5 puntos en la tasa;
ejercicio 3, conteos exactos y ±0,5 puntos en la proporción de pymes; ejercicio 4, solo pistas.
"""


def _estilo_graficas():
    """Colores del curso y letras que se leen en Colab y proyectadas."""
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    plt.rcParams.update({
        "axes.prop_cycle": mpl.cycler(color=["#8A2E9E", "#F46201", "#566B7F"]),
        "figure.figsize": (6.4, 3.6),
        "figure.constrained_layout.use": True,
        "font.size": 13,
        "axes.titlesize": 15,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#94A3B8",
        "axes.labelcolor": "#334155",
        "xtick.color": "#475569",
        "ytick.color": "#475569",
        "text.color": "#1E293B",
    })


_estilo_graficas()

_CACHE = {}   # lo que ya se calculó: los datos escalados y los agrupamientos de referencia
_COLUMNAS = ["edad", "usuarios", "consumo_gb"]


def _datos():
    """Los clientes, sus tres columnas sin escalar y escaladas; o None con aviso."""
    if "datos" in _CACHE:
        return _CACHE["datos"]
    import pandas as pd
    from sklearn.preprocessing import StandardScaler

    try:
        clientes = pd.read_csv("datos/clientes.csv")
    except FileNotFoundError:
        print("✗  No encuentro datos/clientes.csv: ejecuta primero la celda que descarga los datos "
              "(Entorno de ejecución → Ejecutar anteriores)")
        return None
    X = clientes[_COLUMNAS]
    _CACHE["datos"] = {"clientes": clientes, "X": X, "Z": StandardScaler().fit_transform(X)}
    return _CACHE["datos"]


def _sobrantes(sueltos, otros, nombres, ejemplo):
    """Avisa de valores sin nombre o con otro nombre. True si hubo alguno."""
    if sueltos:
        print(f"✗  Pasa los valores por nombre, como en la plantilla: {ejemplo}")
        return True
    if otros:
        print(f"✗  No conozco {', '.join(repr(k) for k in otros)}: usa estos nombres, en "
              f"minúsculas y sin tildes: {', '.join(n + '=' for n in nombres)}")
        return True
    return False


def _seguro(verificador):
    """Red de seguridad: si algo imprevisto falla al revisar, se avisa sin traceback."""
    import functools

    @functools.wraps(verificador)
    def envoltura(*args, **kwargs):
        try:
            return verificador(*args, **kwargs)
        except Exception as error:   # noqa: BLE001 — el estudiante nunca ve un traceback
            print(f"✗  No pude revisar lo que entregaste ({type(error).__name__}). Revisa que cada valor "
                  "sea el resultado de un cálculo y vuelve a ejecutar la celda")
    return envoltura


def _un_numero(x):
    """(valor, None) si x es un número (o algo con un solo número); si no, (None, motivo)."""
    import math
    import numbers
    import numpy as np
    import pandas as pd

    if x is Ellipsis:
        return None, "todavía tiene los puntos suspensivos (...) de la plantilla: reemplázalos por tu cálculo"
    if isinstance(x, pd.DataFrame):
        if x.size != 1:
            forma = f"{x.shape[0]} × {x.shape[1]}" if x.shape[0] <= 10 else "muchas filas"   # no regalar un conteo
            return None, f"es una tabla de {forma}; hace falta un solo número (¿te faltó contar o promediar?)"
        x = x.iloc[0, 0]
    if isinstance(x, (pd.Series, pd.Index, np.ndarray, list, tuple, dict)):
        valores = list(x.values()) if isinstance(x, dict) else list(np.ravel(np.asarray(x, dtype=object)))
        if len(valores) == 0:
            return None, "está vacío"
        if len(valores) > 1:
            if len(valores) == 8000:
                return None, "es un valor por cliente; falta resumirlo en un número"
            cuantos = f"{len(valores)} valores" if len(valores) <= 10 else "muchos valores"   # no regalar un conteo
            return None, f"tiene {cuantos}; hace falta uno solo (¿te faltó contar con len() o sum(), o promediar con mean()?)"
        x = valores[0]
        if x is Ellipsis:
            return None, "todavía tiene los puntos suspensivos (...) de la plantilla: reemplázalos por tu cálculo"
    if x is None:
        return None, "está vacío (None): ¿le asignaste el resultado de un print?"
    if callable(x):
        return None, "es una función, no su resultado: ¿faltaron los paréntesis o los datos?"
    if isinstance(x, (bool, np.bool_)):
        return None, "es True o False; hace falta un número"
    if isinstance(x, str):
        return None, "es texto; escríbelo como número, sin comillas y con punto decimal"
    if hasattr(x, "fit") and hasattr(x, "get_params"):
        return None, "es un modelo, no un número: ¿te faltó calcular algo con él?"
    if not isinstance(x, (numbers.Real, np.number)) or isinstance(x, np.complexfloating):
        return None, f"no es un número real (es {type(x).__name__})"
    try:
        valor = float(x)
    except OverflowError:
        return None, "es un número demasiado grande"
    if math.isnan(valor):
        return None, "salió nan: revisa el cálculo"
    if math.isinf(valor):
        return None, "salió infinito: ¿dividiste entre cero?"
    return valor, None


def _leer(entregados, nombres):
    """Convierte cada valor entregado en número; imprime los problemas. Devuelve {nombre: valor}."""
    valores, problemas = {}, {}
    for nombre in nombres:
        x = entregados[nombre]
        if x is Ellipsis:
            print(f"·  {nombre}: todavía tiene los puntos suspensivos (...) de la plantilla: reemplázalos por tu cálculo")
            continue
        valor, problema = _un_numero(x)
        if problema:
            problemas.setdefault(problema, []).append(nombre)
        else:
            valores[nombre] = valor
    for problema, quienes in problemas.items():   # la misma pista una sola vez
        print(f"✗  {', '.join(quienes)}: {problema}")
    return valores


def _etiquetas(x, nombre):
    """Un arreglo de enteros con una etiqueta por cliente, o None con aviso."""
    import numpy as np
    import pandas as pd

    if x is Ellipsis:
        print(f"·  {nombre}: todavía tiene los puntos suspensivos (...) de la plantilla")
        return None
    if isinstance(x, pd.DataFrame) and x.shape[1] == 1:
        x = x.iloc[:, 0]
    try:
        arreglo = np.asarray(x)
    except Exception:   # noqa: BLE001
        arreglo = None
    if arreglo is None or arreglo.ndim != 1 or len(arreglo) != 8000:
        print(f"✗  {nombre}: tiene que ser el número de grupo de cada uno de los 8000 clientes "
              "(lo que devuelve fit_predict)")
        return None
    if not np.issubdtype(arreglo.dtype, np.integer):
        try:
            if np.all(np.mod(arreglo.astype(float), 1) == 0):
                arreglo = arreglo.astype(int)
            else:
                raise ValueError
        except (TypeError, ValueError):
            print(f"✗  {nombre}: tiene que traer números de grupo enteros (0, 1, 2…), como los que devuelve "
                  "fit_predict")
            return None
    return arreglo


def _grupos(n):
    """«1 grupo», «4 grupos»."""
    return f"{int(n)} grupo" + ("" if int(n) == 1 else "s")


def _coma(x, decimales=3):
    """Un número con coma decimal, para los mensajes."""
    return f"{x:.{decimales}f}".replace(".", ",").replace("-", "−")


def _ari(a, b):
    from sklearn.metrics import adjusted_rand_score
    return adjusted_rand_score(a, b)


# ---------------------------------------------------------------------------------------------
# Los agrupamientos de referencia
# ---------------------------------------------------------------------------------------------

def _kmeans(datos, k, n_init=10):
    from sklearn.cluster import KMeans
    return KMeans(n_clusters=k, n_init=n_init, random_state=2026).fit_predict(datos)


def _ref_siluetas(variante="bien"):
    """Siluetas de k = 2…8: «bien» (Z, n_init=10: la clase), «sin_escalar» (X) o «auto» (una sola corrida).
    Cada variante se calcula la primera vez que se pide: la respuesta correcta solo paga la primera."""
    clave = ("siluetas", variante)
    if clave not in _CACHE:
        from sklearn.metrics import silhouette_score

        d = _datos()
        datos = d["X"] if variante == "sin_escalar" else d["Z"]
        n_init = "auto" if variante == "auto" else 10
        _CACHE[clave] = {k: silhouette_score(datos, _kmeans(datos, k, n_init)) for k in range(2, 9)}
    return _CACHE[clave]


def _ref_grupos():
    """Los 4 grupos de la clase (escalados) y los 4 sin escalar."""
    if "grupos" in _CACHE:
        return _CACHE["grupos"]
    d = _datos()
    _CACHE["grupos"] = {"bien": _kmeans(d["Z"], 4), "sin_escalar": _kmeans(d["X"], 4)}
    return _CACHE["grupos"]


def _perfil(grupo_mask):
    """Una descripción del grupo por sus promedios, sin decir su número."""
    d = _datos()
    X = d["X"][grupo_mask]
    pymes = (d["clientes"]["tipo_cliente"][grupo_mask] == "Pyme").mean()
    return (f"edad media de {_coma(X['edad'].mean(), 0)} años, {_coma(X['usuarios'].mean(), 1)} usuarios "
            f"y {_coma(pymes * 100, 0)} % de pymes")


# ---------------------------------------------------------------------------------------------
# Ejercicio 1 · ¿Cuántos grupos?
# ---------------------------------------------------------------------------------------------

def _siluetas_entregadas(siluetas):
    """{k: valor} desde un diccionario, una serie o una lista de 7 (k = 2…8); o None con aviso."""
    import numpy as np
    import pandas as pd

    if siluetas is Ellipsis:
        print("·  siluetas: todavía tiene los puntos suspensivos (...) de la plantilla")
        return None
    if isinstance(siluetas, pd.Series):
        siluetas = siluetas.to_dict()
    if isinstance(siluetas, (list, tuple, np.ndarray)):
        valores = list(np.ravel(np.asarray(siluetas, dtype=object)))
        if len(valores) != 7:
            print(f"✗  siluetas: tiene {len(valores)} valor{'' if len(valores) == 1 else 'es'}; van 7, uno por cada k de 2 a 8 (mejor, un "
                  "diccionario {k: silueta}, como en la plantilla)")
            return None
        siluetas = dict(zip(range(2, 9), valores))
    if not isinstance(siluetas, dict):
        valor, _ = _un_numero(siluetas)
        if valor is not None:
            print("✗  siluetas: es un solo número; van las 7 siluetas, una por cada k de 2 a 8, en el "
                  "diccionario siluetas")
        else:
            print("✗  siluetas: tiene que ser el diccionario {k: silueta} de la plantilla")
        return None
    salida, faltan = {}, []
    for k in range(2, 9):
        clave = k if k in siluetas else (str(k) if str(k) in siluetas else None)
        if clave is None:
            faltan.append(k)
            continue
        valor, problema = _un_numero(siluetas[clave])
        if problema:
            print(f"✗  siluetas[{k}]: {problema}")
            return None
        salida[k] = valor
    if faltan:
        print(f"✗  siluetas: faltan los k {', '.join(map(str, faltan))}; el ciclo va de 2 a 8 "
              "(range(2, 9) termina en 8)")
        return None
    return salida


@_seguro
def revisar_k(*sueltos, siluetas=..., k_silueta=..., k_gerente=..., **otros):
    """Ejercicio 1: las siluetas de k = 2…8, el k de la silueta más alta y el k para la gerente."""
    nombres = ["siluetas", "k_silueta", "k_gerente"]
    if _sobrantes(sueltos, otros, nombres, "revisar_k(siluetas=siluetas, k_silueta=k_silueta, "
                                           "k_gerente=k_gerente)"):
        return
    if _datos() is None:
        return
    ref = {"bien": _ref_siluetas("bien")}
    bien_todo = True
    s = _siluetas_entregadas(siluetas)
    if s is None:
        bien_todo = False
    else:
        malos = [k for k in s if abs(s[k] - ref["bien"][k]) > 0.005]
        if not malos:
            print("✓  siluetas: bien")
        else:
            bien_todo = False
            if all(abs(s[k] - _ref_siluetas("sin_escalar")[k]) <= 0.005 for k in s):
                print("✗  siluetas: son las de los datos sin escalar (X). Agrupa y mide con los datos "
                      "escalados, Z, en fit_predict y en silhouette_score")
            elif all(abs(s[k] - _ref_siluetas("auto")[k]) <= 0.005 for k in s):
                print("✗  siluetas: son las de una sola corrida de K-Means: pon n_init=10")
            elif all(1 < s[k] <= 100 for k in s):
                print("✗  siluetas: la silueta va de −1 a 1; no la escribas como porcentaje")
            elif all(abs(s[k] - ref["bien"][k]) <= 0.03 for k in s):
                print(f"✗  siluetas: casi; {', '.join(f'k = {k}' for k in malos)} no coinciden. Revisa que "
                      "uses n_init=10 y random_state=2026, como en la plantilla")
            elif any(s[k] > 1 or s[k] < -1 for k in s):
                print("✗  siluetas: una silueta siempre está entre −1 y 1: ¿guardaste la inercia u otra "
                      "cosa? Usa silhouette_score(Z, etiquetas_k)")
            else:
                print("✗  siluetas: no coinciden. Para cada k: entrena con Z, y mide "
                      "silhouette_score(Z, etiquetas_k) con las etiquetas de ese mismo k")
    mejor = max(ref["bien"], key=ref["bien"].get)
    vals = _leer({"k_silueta": k_silueta, "k_gerente": k_gerente}, ["k_silueta", "k_gerente"])
    ks = vals.get("k_silueta")
    if "k_silueta" not in vals:
        bien_todo = False
    elif ks != int(ks) or not 2 <= ks <= 8:
        print("✗  k_silueta: es uno de los k que probaste, un entero de 2 a 8")
        bien_todo = False
    elif int(ks) == mejor:
        print("✓  k_silueta: bien")
    else:
        bien_todo = False
        if int(ks) == 8:
            print("✗  k_silueta: ese es el k más grande que probaste, no el de la silueta más alta. Si usaste "
                  "max(siluetas): sobre un diccionario, max compara las claves; pídele que compare los valores "
                  "con key=siluetas.get")
        elif s is not None and int(ks) == max(s, key=s.get):
            print("✗  k_silueta: es el k de tu silueta más alta, pero tus siluetas no coinciden: corrígelas "
                  "primero")
        else:
            print(f"✗  k_silueta: con k = {int(ks)}, la silueta es {_coma(ref['bien'][int(ks)])}; hay un k con "
                  "una silueta más alta. La pregunta es solo por la silueta, no por lo que conviene")
    kg = vals.get("k_gerente")
    if "k_gerente" not in vals:
        bien_todo = False
    elif kg != int(kg) or kg < 1:
        print("✗  k_gerente: es un número de grupos: un entero positivo")
        bien_todo = False
    elif 3 <= kg <= 5:
        print(f"✓  k_gerente: {int(kg)} grupos caben en lo que la gerente puede atender")
    else:
        bien_todo = False
        if kg < 3:
            print(f"✗  k_gerente: con {_grupos(kg)} le quedaría por debajo de lo que la gerente puede atender "
                  "(de 3 a 5 ofertas distintas): ¿qué se pierde con un corte tan grueso?")
        else:
            print(f"✗  k_gerente: con {_grupos(kg)} se pasa de lo que la gerente puede atender (de 3 a 5 "
                  "ofertas distintas): ¿cuántas ofertas tendría que preparar?")
    if bien_todo:
        print(f"\n¡Listo! La silueta más alta es la de k = {mejor} ({_coma(ref['bien'][mejor])}). Escribe en "
              "«Tus respuestas» por qué recomiendas otro k a la gerente: ¿qué separa el corte de "
              f"{mejor} grupos? (Míralo con perfiles, como en la celda [perfiles].)")


# ---------------------------------------------------------------------------------------------
# Ejercicio 2 · ¿Cancelan igual?
# ---------------------------------------------------------------------------------------------

@_seguro
def revisar_cancelacion(*sueltos, grupos=..., grupo=..., tasa=..., **otros):
    """Ejercicio 2: el grupo (de los 4 de la clase) que más cancela y su tasa de cancelación."""
    import numpy as np

    nombres = ["grupos", "grupo", "tasa"]
    if _sobrantes(sueltos, otros, nombres, "revisar_cancelacion(grupos=grupos, grupo=grupo_mas, "
                                           "tasa=tasa_mas)"):
        return
    d = _datos()
    if d is None:
        return
    g = _etiquetas(grupos, "grupos")
    if g is None:
        return
    distintos = np.unique(g[g >= 0])
    if len(distintos) != 4:
        print(f"✗  grupos: tiene {_grupos(len(distintos))} y el ejercicio es con los 4 de la clase: vuelve a "
              "ejecutar la celda [kmeans4] (la variable grupos cambió)")
        return
    ref = _ref_grupos()
    if _ari(g, ref["bien"]) < 0.95:
        if _ari(g, ref["sin_escalar"]) >= 0.95:
            print("✗  grupos: esos son los grupos sin escalar; usa los de la celda [kmeans4], hechos con Z")
        else:
            print("✗  grupos: no son los 4 grupos de la clase: vuelve a ejecutar la celda [kmeans4]")
        return
    cancela = (d["clientes"]["cancelo"] == "Sí").to_numpy()
    tasas = {int(e): cancela[g == e].mean() for e in distintos}
    el_mas = max(tasas, key=tasas.get)
    vals = _leer({"grupo": grupo, "tasa": tasa}, ["grupo", "tasa"])
    bien_g = bien_t = False
    v = vals.get("grupo")
    if v is not None:
        if v != int(v) or int(v) not in tasas:
            print(f"✗  grupo: es el número de uno de los grupos ({', '.join(map(str, sorted(tasas)))}), no "
                  "su tasa ni su tamaño")
        elif int(v) == el_mas:
            bien_g = True
            print("✓  grupo: bien")
        else:
            print(f"✗  grupo: el grupo {int(v)} (el de {_perfil(g == int(v))}) cancela el "
                  f"{_coma(tasas[int(v)] * 100, 1)} %; hay uno que cancela más")
    t = vals.get("tasa")
    if t is not None:
        objetivo = tasas[el_mas]
        if abs(t - objetivo) <= 0.015:
            bien_t = True
            print("✓  tasa: bien")
        elif 1 < t <= 100 and abs(t / 100 - objetivo) <= 0.015:
            bien_t = True
            print("✓  tasa: bien, aunque la escribiste como porcentaje; como proporción va entre 0 y 1")
        elif abs(t - cancela.mean()) <= 0.002 or abs(t / 100 - cancela.mean()) <= 0.002:
            print("✗  tasa: esa es la tasa de todos los clientes; va la del grupo que más cancela "
                  "(agrupa con .groupby(grupos) antes de promediar)")
        elif t == int(t) and t > 100:
            print("✗  tasa: parece un conteo de clientes; la tasa es una proporción: el promedio de cancela "
                  "dentro del grupo")
        else:
            otro = [e for e, r in tasas.items() if e != el_mas and (abs(t - r) <= 0.002 or abs(t / 100 - r) <= 0.002)]
            if otro:
                print(f"✗  tasa: esa es la tasa del grupo {otro[0]}, no la del que más cancela")
            elif abs(t - (1 - objetivo)) <= 0.002:
                print("✗  tasa: esa es la proporción de los que NO cancelaron: compara con \"Sí\"")
            else:
                print("✗  tasa: no coincide: es el promedio de cancela (True o False) entre los clientes de ese "
                      "grupo")
    if bien_g and bien_t:
        resto = sorted(r for e, r in tasas.items() if e != el_mas)
        print(f"\n¡Listo! El grupo de {_perfil(g == el_mas)} cancela el {_coma(tasas[el_mas] * 100, 1)} %; los "
              f"otros, entre el {_coma(resto[0] * 100, 1)} y el {_coma(resto[-1] * 100, 1)} %. Recuerda que "
              "cancelo no entró al agrupamiento: los grupos no se buscaron para separar a los que cancelan, y aun "
              "así cancelan distinto. Escribe en «Tus respuestas» qué perfil es y qué le propondrías a la gerente.")


# ---------------------------------------------------------------------------------------------
# Ejercicio 3 · Lo raro (DBSCAN)
# ---------------------------------------------------------------------------------------------

def _ref_dbscan():
    """Grupos, ruido y pymes del ruido para eps 0,3 y 0,5: la clase y los errores típicos."""
    if "dbscan" in _CACHE:
        return _CACHE["dbscan"]
    import numpy as np
    from sklearn.cluster import DBSCAN

    d = _datos()
    pyme = (d["clientes"]["tipo_cliente"] == "Pyme").to_numpy()
    ref = {}
    for nombre, datos, ms in (("bien", d["Z"], 20), ("ms5", d["Z"], 5), ("sin_escalar", d["X"], 20)):
        ref[nombre] = {}
        for eps in (0.3, 0.5):
            e = DBSCAN(eps=eps, min_samples=ms).fit_predict(datos)
            r = e == -1
            ref[nombre][eps] = {"grupos": int(e.max() + 1), "ruido": int(r.sum()),
                                "pymes": float(pyme[r].mean()) if r.any() else float("nan"),
                                "pymes_ruido_de_pymes": float(r[pyme].mean())}
    _CACHE["dbscan"] = ref
    return ref


@_seguro
def revisar_ruido(*sueltos, grupos_03=..., ruido_03=..., grupos_05=..., ruido_05=..., pymes_ruido=...,
                  **otros):
    """Ejercicio 3: grupos y ruido de DBSCAN con eps 0,3 y 0,5 (min_samples=20), y las pymes del ruido."""
    nombres = ["grupos_03", "ruido_03", "grupos_05", "ruido_05", "pymes_ruido"]
    if _sobrantes(sueltos, otros, nombres, "revisar_ruido(grupos_03=grupos_03, ruido_03=ruido_03, "
                                           "grupos_05=grupos_05, ruido_05=ruido_05, pymes_ruido=pymes_ruido)"):
        return
    if _datos() is None:
        return
    vals = _leer({"grupos_03": grupos_03, "ruido_03": ruido_03, "grupos_05": grupos_05,
                  "ruido_05": ruido_05, "pymes_ruido": pymes_ruido}, nombres)
    if not vals:
        return
    ref = _ref_dbscan()
    bien = {}
    for eps, sufijo in ((0.3, "03"), (0.5, "05")):
        b = ref["bien"][eps]
        for que in ("grupos", "ruido"):
            nombre = f"{que}_{sufijo}"
            if nombre not in vals:
                continue
            v = vals[nombre]
            bien[nombre] = v == b[que]
            if bien[nombre]:
                print(f"✓  {nombre}: bien")
                continue
            hermano_bien = vals.get(f"ruido_{sufijo}") == b["ruido"]   # si el ruido está bien, min_samples y Z también
            if v != int(v):
                print(f"✗  {nombre}: es un conteo, un número entero" +
                      ("; el ruido va como número de clientes, no como proporción" if que == "ruido" else ""))
            elif que == "grupos" and hermano_bien and v == b["grupos"] + 1:
                print(f"✗  {nombre}: estás contando el ruido (-1) como un grupo más")
            elif que == "grupos" and hermano_bien and v == b["grupos"] - 1:
                print(f"✗  {nombre}: te falta sumar 1: las etiquetas empiezan en 0, así que el número de grupos "
                      "es la etiqueta más alta + 1")
            elif v == ref["ms5"][eps][que]:
                print(f"✗  {nombre}: es lo que sale con min_samples por defecto (5); pon min_samples=20")
            elif v == ref["sin_escalar"][eps][que]:
                print(f"✗  {nombre}: es lo que sale sin escalar; usa Z")
            elif que == "grupos" and v == b["grupos"] + 1:
                print(f"✗  {nombre}: estás contando el ruido (-1) como un grupo más")
            elif que == "ruido" and v == 8000 - b["ruido"]:
                print(f"✗  {nombre}: contaste los que NO son ruido; el ruido es la etiqueta -1")
            elif que == "ruido" and v == ref["bien"][0.5 if eps == 0.3 else 0.3]["ruido"]:
                print(f"✗  {nombre}: es el ruido del otro eps; revisa qué etiquetas usaste")
            elif que == "grupos" and v == ref["bien"][0.5 if eps == 0.3 else 0.3]["grupos"]:
                print(f"✗  {nombre}: es el número de grupos del otro eps; revisa qué etiquetas usaste")
            else:
                print(f"✗  {nombre}: no coincide. Pista: el número de grupos es la etiqueta más alta + 1, y el "
                      "ruido, cuántas etiquetas valen -1")
    p = vals.get("pymes_ruido")
    if p is not None:
        b = ref["bien"][0.5]["pymes"]
        if abs(p - b) <= 0.005:
            bien["pymes_ruido"] = True
            print("✓  pymes_ruido: bien")
        elif 1 < p <= 100 and abs(p / 100 - b) <= 0.005:
            bien["pymes_ruido"] = True
            print("✓  pymes_ruido: bien, aunque la escribiste como porcentaje; como proporción va entre 0 y 1")
        else:
            bien["pymes_ruido"] = False
            q = p / 100 if 1 < p <= 100 else p
            if abs(q - ref["bien"][0.5]["pymes_ruido_de_pymes"]) <= 0.005:
                print("✗  pymes_ruido: esa es la proporción de las pymes que quedaron como ruido; la pregunta "
                      "es al revés: de los clientes del ruido, ¿qué proporción son pymes?")
            elif abs(q - ref["bien"][0.3]["pymes"]) <= 0.005:
                print("✗  pymes_ruido: esa es la del ruido con eps 0,3; va la de eps 0,5")
            elif p == int(p) and p > 100:
                print("✗  pymes_ruido: parece un conteo; va la proporción (entre 0 y 1)")
            else:
                print("✗  pymes_ruido: no coincide: filtra a los clientes del ruido (etiquetas_05 == -1) y "
                      "promedia si su tipo_cliente es \"Pyme\"")
    if len(bien) == 5 and all(bien.values()):
        r3, r5 = ref["bien"][0.3], ref["bien"][0.5]
        print(f"\n¡Listo! Con eps 0,5 hay {r5['grupos']} grupo y {r5['ruido']} clientes de ruido, casi todos pymes; "
              f"con 0,3, {r3['grupos']} grupos y {r3['ruido']} de ruido. Mira en «Tus respuestas» qué tienen en "
              "común los clientes de cada grupo de eps 0,3 (la pista está en el enunciado).")


# ---------------------------------------------------------------------------------------------
# Ejercicio 4 (opcional) · Otra columna: solo pistas
# ---------------------------------------------------------------------------------------------

@_seguro
def revisar_otra_columna(*sueltos, columnas=..., grupos5=..., **otros):
    """Ejercicio 4: agrupa con plan_mbps además de las tres columnas, con k = 5. Solo da pistas."""
    import numpy as np

    nombres = ["columnas", "grupos5"]
    if _sobrantes(sueltos, otros, nombres, "revisar_otra_columna(columnas=columnas5, grupos5=grupos5)"):
        return
    d = _datos()
    if d is None:
        return
    if columnas is Ellipsis:
        print("·  columnas: todavía tiene los puntos suspensivos (...) de la plantilla")
        return
    if isinstance(columnas, str):
        columnas = [columnas]
    try:
        lista = list(columnas)
    except TypeError:
        print("✗  columnas: tiene que ser una lista de nombres de columnas")
        return
    if any(c is Ellipsis for c in lista):
        print("·  columnas: todavía tiene los puntos suspensivos (...): cámbialos por \"plan_mbps\", con comillas")
        return
    faltan = [c for c in _COLUMNAS + ["plan_mbps"] if c not in lista]
    sobran = [c for c in lista if c not in _COLUMNAS + ["plan_mbps"]]
    if faltan or sobran:
        print("✗  columnas: van las tres de la clase más \"plan_mbps\"" +
              (f"; faltan {', '.join(map(str, faltan))}" if faltan else "") +
              (f"; sobran {', '.join(map(str, sobran))}" if sobran else ""))
        return
    print("✓  columnas: las cuatro")
    g5 = _etiquetas(grupos5, "grupos5")
    if g5 is None:
        return
    k = len(np.unique(g5))
    if k != 5:
        print(f"✗  grupos5: tiene {_grupos(k)}; el ejercicio pide n_clusters=5")
        return
    from sklearn.preprocessing import StandardScaler
    clientes = d["clientes"]
    datos4 = clientes[_COLUMNAS + ["plan_mbps"]]
    ref = _kmeans(StandardScaler().fit_transform(datos4), 5)
    if _ari(g5, ref) < 0.95:
        if _ari(g5, _kmeans(datos4, 5)) >= 0.95:
            print("✗  grupos5: esos grupos salen sin escalar; escala las cuatro columnas con StandardScaler")
        elif _ari(g5, _kmeans(d["Z"], 5)) >= 0.95:
            print("✗  grupos5: esos grupos salen de las tres columnas de antes (Z); agrupa con Z5, que ya "
                  "trae plan_mbps")
        else:
            print("✗  grupos5: no coinciden con los de la clase: revisa n_init=10 y random_state=2026")
        return
    print("✓  grupos5: 5 grupos, con las cuatro columnas escaladas")
    print("\nPistas para leer la tabla (sin la respuesta):\n"
          "·  Busca la fila de los 4 grupos de antes cuyos clientes se reparten casi por mitades entre dos "
          "columnas (en las otras filas, una sola columna se lleva dos tercios o más).\n"
          "·  Compara el plan_mbps medio de esas dos columnas: "
          "clientes.groupby(grupos5)[\"plan_mbps\"].mean()\n"
          "·  ¿Te sirve ese corte para una oferta? Escríbelo en «Tus respuestas».")

# Descarga datos/clientes.csv: los mismos 8000 clientes de las sesiones 1 y 2
ruta = descargar_datos("datos")

# Parte 1 · Agrupamiento: ¿qué tipos de clientes tenemos?

## 1 · Hoy no hay etiqueta

En la sesión 2, cada cliente traía la respuesta (cuánto consumió, si canceló) y el modelo aprendía a
predecirla: aprendizaje **supervisado**. Hoy la gerente de experiencia del cliente pregunta otra cosa:
*¿qué tipos de clientes tenemos?* Nadie ha etiquetado a los clientes por tipo, así que no hay una
respuesta correcta con la cual comparar. Es aprendizaje **no supervisado**: el **agrupamiento**
(*clustering*) busca grupos de clientes parecidos entre sí y distintos de los demás.

**Repaso: lo nuevo de pandas y numpy que aparece hoy.** No hace falta memorizarlo; vuelve aquí cuando lo veas.

| Escribes | Qué hace |
|---|---|
| `modelo.fit_predict(Z)` | Entrena y devuelve, de una vez, el número de grupo de cada fila |
| `serie.value_counts().sort_index()` | Cuántas filas tiene cada valor, ordenado por el valor |
| `tabla.groupby(grupos).agg(nuevo=("col", "mean"))` | Una fila por grupo, con una columna `nuevo` que resume `col` |
| `lambda t: (t == "Pyme").mean()` | Una función de una línea: la proporción de `"Pyme"` en `t` |
| `tabla.apply(f, axis=1)` | Aplica la función `f` a cada fila |
| `pd.crosstab(a, b)` | Cuenta cuántas filas hay en cada combinación de `a` y `b` |
| `arreglo[:, [0, 2]]` | De un arreglo de numpy, todas las filas y las columnas 0 y 2 |
| `(etiquetas == -1).sum()` | Cuántas etiquetas valen −1: True cuenta como 1 |
| `etiquetas.max() + 1` | Con etiquetas 0, 1, 2…, cuántos grupos hay |
| `clientes.loc[mascara, "col"]` | La columna `col`, solo en las filas donde `mascara` es True |
| `serie.idxmax()` | La etiqueta (el índice) del valor más alto |
| `max(d, key=d.get)` | La clave del diccionario `d` con el valor más alto |

In [ ]:
# [cargar]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

clientes = pd.read_csv("datos/clientes.csv")
# Colores de los grupos; el último, gris, es para el ruido (-1) de DBSCAN
COLORES = np.array(["#8A2E9E", "#F46201", "#566B7F", "#0E7490", "#94A3B8"])
np.set_printoptions(legacy="1.25")   # números sin np.float64(...) al mostrarlos
print(f"scikit-learn {sklearn.__version__}")
clientes.shape

## 2 · Parecido es distancia, y la distancia depende de la escala

Agrupamos con tres columnas que dicen **quién es el cliente y cuánto usa la conexión**:

| Columna | Qué registra | Unidad |
|---|---|---|
| `edad` | Edad del titular | años |
| `usuarios` | Personas que usan la conexión (declarado al contratar) | conteo |
| <code>consumo&#95;gb</code> | Datos consumidos al mes, en promedio | GB |

`cancelo` **no** entra: no buscamos a los que cancelan, sino tipos de clientes. (El diccionario completo
está en el notebook de la sesión 1.)

In [ ]:
# [columnas]
columnas = ["edad", "usuarios", "consumo_gb"]
X = clientes[columnas]
X.agg(["mean", "std", "min", "max"]).round(1)

Un cliente es «parecido» a otro si está **cerca**: la distancia en línea recta se calcula con las
diferencias de cada columna, elevadas al cuadrado. Pero el consumo va de 14 a 2500 GB y los usuarios, de 1 a 14: sin escalar, la
distancia la decide casi solo el consumo. Como con KNN en la sesión 2, **escalamos**: cada columna queda
en desviaciones estándar (`StandardScaler`).

Una diferencia con la sesión 2: aquí no hay entrenamiento ni prueba, porque no hay una respuesta que
predecir y medir. El escalador aprende la media y la desviación de los 8000 clientes.

In [ ]:
# [escala]
from sklearn.preprocessing import StandardScaler
escalador = StandardScaler()
Z = escalador.fit_transform(X)            # cada columna, en desviaciones
print("Medias:", Z.mean(axis=0).round(2) + 0.0)   # + 0.0: sin «−0»
print("Desviaciones:", Z.std(axis=0).round(2))

## 3 · K-Means (k-medias)

Imagina que Yarumo abre **k puntos de atención** en una ciudad (un mapa de perfiles: sus ejes son la edad,
los usuarios y el consumo, no calles) y cada cliente va al más cercano. Después,
cada punto se muda al **centro** de los clientes que le tocaron, y se repite: los clientes cambian de punto
si otro les quedó más cerca, los puntos se vuelven a mudar… hasta que nadie cambia. Cada punto es un
**centro del grupo** (centroide) y sus clientes, un **grupo** (*cluster*).

La celda siguiente lo hace a mano con 300 clientes y solo dos columnas (edad y consumo, escaladas), para
poder dibujarlo. Si la lees, verás las dos operaciones: `asignar` (cada cliente al centro más cercano) y
`mudar` (cada centro al promedio de sus clientes).

In [ ]:
# [pasos]
def asignar(puntos, centros):
    """El número del centro más cercano a cada punto."""
    distancias = ((puntos[:, None, :] - centros[None, :, :]) ** 2).sum(axis=2)
    return distancias.argmin(axis=1)

def mudar(puntos, cercano, k):
    """Cada centro pasa al promedio de los puntos que le tocaron."""
    return np.array([puntos[cercano == j].mean(axis=0) for j in range(k)])

def dibujar(ax, puntos, cercano, centros, titulo):
    color = "#94A3B8" if cercano is None else COLORES[cercano]
    ax.scatter(puntos[:, 0], puntos[:, 1], s=9, c=color)
    ax.scatter(centros[:, 0], centros[:, 1], s=180, marker="X",
               c="#1E293B", edgecolors="white")
    ax.set_title(titulo, fontsize=13)

muestra = Z[:300][:, [0, 2]]               # edad y consumo, escalados
rng = np.random.default_rng(2026)
centros = muestra[rng.choice(300, size=3, replace=False)]  # al azar
fig, ejes = plt.subplots(1, 4, figsize=(13, 3.6), sharex=True,
                         sharey=True)
dibujar(ejes[0], muestra, None, centros, "1 · Tres centros al azar")
cercano = asignar(muestra, centros)
dibujar(ejes[1], muestra, cercano, centros, "2 · Cada uno al más cercano")
centros = mudar(muestra, cercano, 3)
dibujar(ejes[2], muestra, cercano, centros, "3 · Los centros se mudan")
for vuelta in range(10):                   # repetir (aquí bastan diez vueltas)
    cercano = asignar(muestra, centros)
    centros = mudar(muestra, cercano, 3)
dibujar(ejes[3], muestra, cercano, centros, "4 · Diez vueltas después")
ejes[0].set_ylabel("Consumo (escalado)")
for ax in ejes:
    ax.set_xlabel("Edad (escalada)")
plt.show()

K-Means busca los centros que dejan la menor **inercia**: la suma de los **cuadrados** de las distancias
de cada cliente a su centro. El resultado depende de dónde empiecen los centros, así que scikit-learn
arranca con *k-means++* (centros iniciales lejos unos de otros), **repite** el proceso `n_init` veces y
se queda con la corrida de menor inercia. Ojo: en la versión de Colab, `n_init="auto"` hace **una sola**
corrida; por eso hoy siempre escribimos `n_init=10` y la semilla `random_state=2026`.

In [ ]:
# [kmeans4]
from sklearn.cluster import KMeans
km4 = KMeans(n_clusters=4, n_init=10, random_state=2026)
grupos = km4.fit_predict(Z)           # el número de grupo de cada cliente
print(f"Inercia: {km4.inertia_:.1f}")
pd.Series(grupos).value_counts().sort_index()

Los números de los grupos (0, 1, 2, 3) **no significan nada**: son solo nombres. Con otra semilla o con
otra versión de scikit-learn, el mismo grupo puede salir con otro número. Por eso, más adelante, cada
grupo se nombra por su **perfil**.

## 4 · ¿Cuántos grupos?

K-Means no elige k: hay que dárselo. Dos ayudas:

- **El método del codo** (*elbow method*). La inercia siempre baja al subir k (con 8000 grupos sería 0). Se busca el punto donde deja
  de bajar mucho. Es una **heurística**, no una regla.
- **El coeficiente de silueta.** Para cada cliente compara *a*, su distancia media a los de su grupo, con *b*, su distancia
  media al grupo vecino más cercano: $s = (b - a) / \max(a, b)$. Va de −1 a 1; cerca de 1, el cliente está
  bien metido en su grupo; cerca de 0, está en la frontera. La silueta del agrupamiento es el promedio.

La **línea base** es repartir a los clientes **al azar** en 4 grupos: su silueta queda cerca de 0.

In [ ]:
# [codo_silueta]
from sklearn.metrics import silhouette_score
ks = range(2, 9)
inercias, siluetas = [], []
for k in ks:
    modelo = KMeans(n_clusters=k, n_init=10, random_state=2026).fit(Z)
    inercias.append(modelo.inertia_)
    siluetas.append(silhouette_score(Z, modelo.labels_))
azar = np.random.default_rng(2026).integers(0, 4, size=len(Z))
silueta_azar = silhouette_score(Z, azar)  # línea base: 4 grupos al azar
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.9))
ax1.plot(ks, inercias, "o-", color="#8A2E9E", linewidth=2.5)
ax1.set_title("El codo: la inercia baja cada vez menos")
ax1.set_xlabel("Número de grupos (k)")
ax1.set_ylabel("Inercia")
ax2.plot(ks, siluetas, "o-", color="#F46201", linewidth=2.5,
         label="K-Means")
ax2.axhline(silueta_azar, color="#566B7F", linestyle="--",
            label="4 grupos al azar")
ax2.set_title("La silueta: más alta, grupos más separados")
ax2.set_xlabel("Número de grupos (k)")
ax2.set_ylabel("Silueta media")
ax2.legend(frameon=False)
plt.show()

### La trampa de no escalar

¿Y si no escalamos? Antes de ejecutar la celda, predice: **¿la silueta sube o baja?**

In [ ]:
# [sin_escalar]
km_crudo = KMeans(n_clusters=4, n_init=10, random_state=2026)
grupos_crudos = km_crudo.fit_predict(X)        # sin escalar
print(f"Silueta sin escalar: {silhouette_score(X, grupos_crudos):.3f}")
print(f"Silueta escalada:    {silhouette_score(Z, grupos):.3f}")
X.groupby(grupos_crudos).agg(["min", "max"]).round(0)

Sin escalar, la silueta **sube**, y los grupos son peores: casi solo ordenan por consumo (cada grupo es
una franja de GB), y sus clientes no se parecen entre sí en edad ni en usuarios. La silueta mide
distancias con las columnas que le das; no sabe si esas columnas son comparables. **La silueta ayuda,
pero no decide.**

## 5 · Los cuatro grupos, por su perfil

Para la gerente, un grupo sirve si se puede **describir**. Resumimos cada uno con sus promedios.

In [ ]:
# [perfiles]
perfiles = clientes.groupby(grupos).agg(
    clientes=("edad", "size"),
    edad=("edad", "mean"),
    usuarios=("usuarios", "mean"),
    consumo_gb=("consumo_gb", "mean"),
    pymes=("tipo_cliente", lambda t: (t == "Pyme").mean()))
perfiles.round({"edad": 1, "usuarios": 1, "consumo_gb": 0, "pymes": 3})

Cada grupo se nombra por su perfil, no por su número. La función `nombrar` lo hace con reglas que
cualquiera puede leer: si cambia el número, el nombre no cambia. Describimos clientes, no juzgamos
personas.

In [ ]:
# [nombres]
def nombrar(fila):
    """Un nombre por el perfil del grupo, no por su número."""
    if fila["pymes"] > 0.5:
        return "Pymes con muchos usuarios"
    if fila["edad"] >= 55:
        return "Titulares mayores, 1 o 2 personas"
    if fila["usuarios"] >= 3:
        return "Hogares de 3 a 6 personas"
    return "Titulares jóvenes, hogares pequeños"

nombre = perfiles.apply(nombrar, axis=1)
nombre

**¿Cancelan igual?** `cancelo` no entró al agrupamiento. Si los grupos cancelan distinto, el agrupamiento
encontró algo que la gerente puede usar. Lo calculas tú en el ejercicio 2.

## 6 · Agrupamiento jerárquico

El **jerárquico aglomerativo** empieza con cada cliente solo y une, paso a paso, los dos grupos más
cercanos, hasta que queda uno. El **dendrograma** dibuja esas uniones: la altura de cada unión es qué tan
distintos eran los grupos que se unieron. **Cortarlo a una altura** da k grupos. Con el criterio de
**Ward**, cada unión es la que menos aumenta la suma de cuadrados dentro de los grupos, un objetivo
parecido al de K-Means. Lo dibujamos con 200 clientes al azar: con 8000 no se leería.

In [ ]:
# [jerarquico]
from scipy.cluster.hierarchy import (dendrogram, linkage,
                                     set_link_color_palette)
rng = np.random.default_rng(2026)
muestra200 = Z[rng.choice(len(Z), size=200, replace=False)]
uniones = linkage(muestra200, method="ward")   # de abajo hacia arriba
corte = (uniones[-4, 2] + uniones[-3, 2]) / 2  # la altura que deja 4 grupos
set_link_color_palette(list(COLORES[:4]))
fig, ax = plt.subplots(figsize=(11, 3.9))
dendrogram(uniones, no_labels=True, color_threshold=corte,
           above_threshold_color="#94A3B8", ax=ax)
ax.axhline(corte, color="#1E293B", linestyle="--", linewidth=1)
ax.set_title("Cortar el dendrograma a una altura da 4 grupos")
ax.set_ylabel("Altura de la unión (Ward)")
ax.set_xlabel("200 clientes al azar")
plt.show()

El criterio de **enlace** (*linkage*) para unir importa. El **enlace simple** (*single*) une los dos grupos
que tienen los clientes más cercanos entre sí. Antes de ejecutar, predice: con los 8000 clientes y 4 grupos,
**¿cuántos clientes tendrá el grupo más grande con el enlace simple?**

In [ ]:
# [encadena]
from sklearn.cluster import AgglomerativeClustering
for enlace in ["ward", "single"]:
    jerarquico = AgglomerativeClustering(n_clusters=4, linkage=enlace)
    etiquetas_j = jerarquico.fit_predict(Z)
    tamanos = sorted(np.bincount(etiquetas_j).tolist(), reverse=True)
    print(f"{enlace:>6}: {tamanos}")

Con el enlace simple, basta una cadena de clientes parecidos para pegarlo todo: el grupo más grande se queda
con casi todos. Ward da tamaños parecidos.

## 7 · DBSCAN: grupos por densidad, y el ruido

Piensa en una plaza llena. En el centro de una zona densa, cada persona tiene, a menos de un paso (`eps`),
al menos `min_samples` personas cerca, contándose a sí misma; quien está en el borde entra al grupo si
toca una zona densa, y quien está solo, lejos de todos, es **ruido**. DBSCAN no pide k: lo
encuentra. Y no obliga a nadie a entrar en un grupo: el ruido sale con la etiqueta `-1`.

Primero, un ejemplo de juguete: dos **lunas** entrelazadas (datos inventados con `make_moons`).

In [ ]:
# [lunas]
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons
lunas, _ = make_moons(n_samples=400, noise=0.08, random_state=2026)
por_kmeans = KMeans(n_clusters=2, n_init=10,
                    random_state=2026).fit_predict(lunas)
por_dbscan = DBSCAN(eps=0.25, min_samples=20).fit_predict(lunas)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.9), sharey=True)
ax1.scatter(lunas[:, 0], lunas[:, 1], s=12, c=COLORES[por_kmeans])
ax1.set_title("K-Means parte las lunas por la mitad")
ax2.scatter(lunas[:, 0], lunas[:, 1], s=12, c=COLORES[por_dbscan])
ax2.set_title("DBSCAN sigue cada luna")
plt.show()

K-Means supone grupos redondos alrededor de un centro; DBSCAN sigue la forma que tenga la zona densa.
(En `COLORES[etiquetas]`, el ruido, `-1`, toma el último color: el gris.)

¿Y en Yarumo? Con nuestras tres columnas escaladas, `eps=0.5` y `min_samples=20`. Antes de ejecutar,
predice: **¿quiénes serán el ruido?**

In [ ]:
# [dbscan]
db = DBSCAN(eps=0.5, min_samples=20)
etiquetas = db.fit_predict(Z)
ruido = etiquetas == -1                     # True si quedó como ruido
print(f"Grupos: {etiquetas.max() + 1} · ruido: {ruido.sum()} clientes")
clientes.groupby(ruido)[columnas].mean().round(1).rename(
    index={False: "el resto", True: "el ruido"})

Con estos datos, DBSCAN no sirve para hacer segmentos: deja casi a todos en un solo grupo. Sirve para otra
cosa: **encontrar lo atípico**, los clientes que no se parecen a nadie. Con `eps` más pequeño pasa algo curioso:
lo ves en el ejercicio 3.

**El método, las columnas y k se eligen por la decisión.** La silueta, el codo y el dendrograma ayudan;
ninguno decide solo.

---
## Ejercicio 1 · ¿Cuántos grupos? (en clase)

Calcula la **silueta** de K-Means para k de 2 a 8, con los datos **escalados** (`Z`), `n_init=10` y la
semilla 2026. Entrega:

- `siluetas`: el diccionario con la silueta de cada k;
- `k_silueta`: el k con la silueta más alta;
- `k_gerente`: el k que le recomiendas a la gerente, sabiendo que **puede atender de 3 a 5 ofertas
  distintas** (una por grupo).

Pista: la celda [codo_silueta]. Cambia cada `...`; el verificador te dice qué está bien y, en lo demás, te
da una pista. Después justifica tu `k_gerente` en «Tus respuestas»: ¿qué te dicen el codo, la silueta y la
gerente?

In [ ]:
# [ej1]
siluetas = {}
for k in range(2, 9):
    modelo = KMeans(n_clusters=k, n_init=10, random_state=2026)
    etiquetas_k = modelo.fit_predict(...)      # con los datos escalados
    siluetas[k] = ...                          # silhouette_score(...)
    print(k, siluetas[k])
k_silueta = ...           # el k de la silueta más alta (máximo del diccionario)
k_gerente = ...                                # el que le recomiendas
revisar_k(siluetas=siluetas, k_silueta=k_silueta, k_gerente=k_gerente)

## Ejercicio 2 · ¿Cancelan igual? (en clase)

Con los cuatro grupos de la clase (`grupos`, de la celda [kmeans4]), calcula la **tasa de cancelación de
cada grupo**: la proporción de clientes con `cancelo == "Sí"`. Entrega el **número del grupo** que más
cancela y **su tasa** (como proporción, entre 0 y 1). Pista: en la sesión 1, la tasa era el promedio de una
columna de True y False; aquí, ese promedio se hace por grupo (`.groupby(grupos).mean()`).

Después responde abajo: ¿qué **perfil** tiene ese grupo (míralo en [perfiles] y [nombres])? ¿Qué le
propondrías a la gerente?

In [ ]:
# [ej2]
cancela = clientes["cancelo"] == "Sí"         # True si canceló
tasas = ...                                    # la tasa de cada grupo
print(tasas)
grupo_mas = ...                                # el número del que más cancela
tasa_mas = ...                                 # su tasa
revisar_cancelacion(grupos=grupos, grupo=grupo_mas, tasa=tasa_mas)

## Ejercicio 3 · Lo atípico (trabajo independiente)

Corre DBSCAN sobre `Z` con `min_samples=20` y dos valores de `eps`: `0.3` y `0.5`. Para cada uno, cuenta
**cuántos grupos** encontró (el ruido, `-1`, no es un grupo) y **cuántos clientes quedaron como ruido**. Con
`eps=0.5`, calcula además **qué proporción del ruido son pymes**. Pista: la celda [dbscan].

Después responde abajo: con `eps=0.3`, ¿qué tienen en común los clientes de cada grupo? (Mira
`pd.crosstab(etiquetas_03, clientes["usuarios"])`.) ¿Por qué pasa eso?

In [ ]:
# [ej3]
etiquetas_03 = DBSCAN(eps=0.3, min_samples=...).fit_predict(Z)
etiquetas_05 = DBSCAN(eps=..., min_samples=...).fit_predict(Z)
grupos_03 = ...                  # cuántos grupos con eps 0.3, sin el ruido
ruido_03 = ...                   # cuántos clientes son ruido con eps 0.3
grupos_05 = ...
ruido_05 = ...
pymes_ruido = ...                # proporción de pymes en el ruido (eps 0.5)
revisar_ruido(grupos_03=grupos_03, ruido_03=ruido_03, grupos_05=grupos_05,
              ruido_05=ruido_05, pymes_ruido=pymes_ruido)

## Ejercicio 4 (opcional) · Otra columna

Agrega `plan_mbps` (la velocidad contratada) a las tres columnas, escala de nuevo y agrupa con **k = 5**.
La tabla cruza los 4 grupos de la clase (filas, por su nombre) con los 5 nuevos (columnas). ¿Cuál de los grupos de antes
se parte en dos? ¿Por qué se parte? Pista: compara el `plan_mbps` medio de los grupos nuevos. Este
verificador solo da pistas.

In [ ]:
# [ej4]
columnas5 = columnas + [...]                   # agrega plan_mbps
Z5 = StandardScaler().fit_transform(clientes[columnas5])
grupos5 = KMeans(n_clusters=..., n_init=10,
                 random_state=2026).fit_predict(Z5)
print(pd.crosstab(nombre[grupos].values, grupos5, rownames=["antes"],
                  colnames=["con plan_mbps"]))
revisar_otra_columna(columnas=columnas5, grupos5=grupos5)

---
# Parte 2 · Nube: riesgos, privacidad y dónde vive el modelo

## Para Yarumo · Lo que llevas a las salas (sin código)

Esta parte se hace con las diapositivas de la sesión y pasa directo al **plan de nube** del proyecto. Con
tu pareja, escribe una línea por fila. *Doble clic para escribir aquí.*

| Decisión | Lo que eligen | Por qué, y con qué fuente |
|---|---|---|
| Región de Azure para los datos y el modelo | … | … |
| Base legal: ¿transmisión o transferencia? ¿Con qué contrato? | … | … |
| Modelo de servicio de nube para el modelo (máquina virtual, App Service o un punto de conexión administrado) | … | … |
| Riesgo 1 y el control que lo mitiga (de la S2 o de hoy) | … | … |
| Riesgo 2 y el control que lo mitiga | … | … |

---
### Tus respuestas

*Doble clic para escribir aquí.*

1. Ejercicio 1: la silueta más alta es la de k = …; le recomiendo a la gerente k = … porque … (¿qué dice el
   codo?, ¿qué separa el corte de 2 grupos?) …
2. Ejercicio 2: el grupo que más cancela es el de … (su perfil, no su número), con una tasa de …; los demás
   cancelan entre … y … Le propondría a la gerente … ¿Por qué es interesante si `cancelo` no entró al
   agrupamiento? …
3. Ejercicio 3: con `eps` 0,3 salen … grupos y … clientes de ruido; con 0,5, … y … El ruido de 0,5 es sobre
   todo de … Con 0,3, cada grupo tiene … ¿Por qué? …
4. Ejercicio 4 (opcional): con `plan_mbps`, el grupo de … se parte en … y … porque …

### Autoevaluación (respóndela en el cierre de la sesión)

¿Verdadero o falso? Explica cada una.

1. «Con una máquina virtual, Yarumo parchea su sistema operativo». …
2. «Si K-Means usa las columnas sin escalar, pasar el consumo de GB a MB no cambia los grupos». …
3. «Si Microsoft guarda en Mexico Central los datos de Yarumo, por cuenta de Yarumo y con contrato, pasa a ser
   el responsable de esos datos». …
4. «Una silueta alta no basta para elegir el número de grupos». …

---
## La respuesta a la gerencia

**¿Qué tipos de clientes tenemos?** Con la edad, los usuarios y el consumo escalados, K-Means encuentra cuatro
perfiles que se pueden describir: titulares jóvenes de hogares pequeños, hogares de tres a seis personas,
titulares mayores de hogares de una o dos personas y pymes con muchos usuarios. No cancelan igual, aunque la cancelación no
entró al agrupamiento. La silueta prefería dos grupos, pero ese corte casi solo aparta a las pymes: el número de
grupos lo decide lo que la gerente puede hacer con ellos.

**¿Llevamos los datos y el modelo a Azure, y cómo?** Si Microsoft trata los datos por cuenta de Yarumo, es una
transmisión con contrato; la región se elige mirando dónde quedan los datos; el modelo puede vivir en una
máquina virtual (IaaS) o en un servicio administrado (PaaS), y cada opción reparte distinto lo que Yarumo
administra. Según la encuesta de la Cloud Security Alliance de 2026, el riesgo que más pesa hoy es la
identidad: quién entra y con qué permisos.